# Изкуствено увеличаване на тренировъчните данните
В тази тетрадка е кодът, който съм използвал, за да "обоготя" тренировъчното множество с тези поеми, които се римуват най-много.


In [2]:
from accentor import *
import string
from lev_dist import lev_dist
import random

In [3]:
def predict_stress(model, word):
    model.eval()
    device = next(model.parameters()).device
    word_indices = [char2ind.get(c, char2ind[unkChar]) for c in word]
    
    input_tensor = torch.tensor(word_indices, dtype=torch.long, device=device).unsqueeze(1)
    
    with torch.no_grad():
        logits = model(input_tensor)
        probs = torch.sigmoid(logits).squeeze(1)
    
    stress_string = ''.join(['1' if p > 0.5 else '0' for p in probs.cpu().numpy()])
    stress_probs = probs.cpu().numpy().tolist()
    
    return stress_string, stress_probs


model = StressTransformer(char2ind, d_model=64, nhead=4, num_layers=2)
model.load_state_dict(torch.load('/workspace/Neural Poet/models/stress_transformer14.pth'))
model.to(torch.device('cuda'))
model.eval()

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")
/tmp/ipykernel_17317/2705237874.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unles

StressTransformer(
  (embedding): Embedding(66, 64, padding_idx=3)
  (pos_encoder): PositionalEncoding()
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=64, out_features=1, bias=True)
)

Зареждаме корпусът, който вече е почистен от проблемни стихотворения.

In [34]:
with open('./data/cleaned_corpus.txt', 'r') as f:
    corpus = f.read().split('@')

Взимаме последните думи от всеки ред и ги записваме в last_words.

In [35]:
import re

last_words = []
for poem in corpus:
	poem = poem.split('\n')
	poem_last_words = []
	for line in poem:
		if line.strip():
			text = line.split()[-1]
			text = re.sub(r'[^\w\s]', '', text).lower()
			poem_last_words.append(text)
	last_words.append(poem_last_words)


Генерираме техните ударения.

In [36]:
keys = []
for poem_last_words in last_words:
	poem_keys = []
	for word in poem_last_words:
		if len(word) == 0:
			continue
		stress, _ = predict_stress(model, word)
		i = stress.find('1')
		if i != -1:
			poem_keys.append(word[i:])
	keys.append(poem_keys)


Пресмятаме метриката за рима за всяка поема.

In [37]:
poem_scores = []
for poem_keys in keys:
	poem_score = 0
	count = 0
	for i in range(0, len(poem_keys) - 1, 2):
		k1 = poem_keys[i]
		k2 = poem_keys[i + 1]
		if k1[0] != k2[0]:
			poem_score += 2		
		poem_score += lev_dist(k1, k2)	
		count += 1
	if count == 0: poem_score = 5
	else: poem_score /= count
	poem_scores.append(poem_score)


In [38]:
poem_scores_sorted = sorted(enumerate(poem_scores), key=lambda x: x[1])

In [39]:
poem_scores_sorted[1]

(3917, 0.0)

* Повтаряме в тренировъчното множество точно тези поеми с най-добра метрика за римуване. 
* Внимаваме с това да разбъркаме данните преди да ги запишем - досега са били сортирани.
* Не повтаряме поеми в множеството за тестване. С него искаме да мерим перплексия.

In [50]:
good_poems = list(map(lambda x: corpus[x[0]], poem_scores_sorted[:1500]))
second_poems = list(map(lambda x: corpus[x[0]], poem_scores_sorted[1500:2500]))
third_poems = list(map(lambda x: corpus[x[0]], poem_scores_sorted[2500:]))

In [58]:
def train_test_split(poems, test_ratio=0.1):
    poems_copy = poems.copy()
    random.shuffle(poems_copy)
    split_idx = int(len(poems_copy) * (1 - test_ratio))
    return poems_copy[:split_idx], poems_copy[split_idx:]

good_train, good_test = train_test_split(good_poems)
second_train, second_test = train_test_split(second_poems)
third_train, third_test = train_test_split(third_poems)

test_dataset = []
test_dataset.extend(good_test)
test_dataset.extend(second_test)
test_dataset.extend(third_test)

train_dataset = []
train_dataset.extend(good_train * 5)      
train_dataset.extend(second_train * 2)    
train_dataset.extend(third_train)         

random.shuffle(train_dataset)
random.shuffle(test_dataset)

In [60]:
with open('./data/test_dataset_cu.txt', 'w') as f:
	f.write('@'.join(test_dataset))
with open('./data/train_dataset_cu.txt', 'w') as f:
	f.write('@'.join(train_dataset))
